### NexaCorp RAG Chatbot

**Import necessary libraries**

In [9]:
# import necessary libraries
import re
import json
import faiss
import uuid
from dotenv import load_dotenv
from langchain_community.document_loaders import UnstructuredWordDocumentLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain.chat_models import init_chat_model

In [ ]:
# load environment variables
load_dotenv()

True

**Load the document**

In [11]:
# load the required document
file_path = "NexaCorp_Enterprise_Policy_Handbook_v5.2.docx"
try:
    loader = UnstructuredWordDocumentLoader(
        file_path,
        mode="elements"                                 # breaks document into structured chunks based on layout
    )
    docs = loader.load()
except Exception as e:
    raise RuntimeError(f"Failed to load document: {file_path}") from e

In [12]:
# types of elements in document
from collections import Counter
categories = Counter(doc.metadata.get("category", "Unknown") for doc in docs)
print(categories)

Counter({'Title': 150, 'NarrativeText': 125, 'ListItem': 113, 'UncategorizedText': 64, 'Table': 43, 'PageBreak': 33, 'Header': 1, 'Footer': 1})


**Data Cleaning**

In [13]:
# use regular expressions to remove header/footer noise
def is_header_footer(text):
    NOISE_PATTERNS = [
        r"Version\s+\d+[\.\d]*\s*\|",
        r"Internal Restricted",
        r"Not for External Distribution",
        r"©\s*20\d{2}",
        r"CONFIDENTIAL",
        r"Policy.*Handbook"
    ]
    # checks if patterns exist anywhere in the text and returns True if atleast one pattern matches
    return any(re.search(p, text, re.IGNORECASE) for p in NOISE_PATTERNS)

# uses regular expression to check if line belongs to TOC or not
def is_toc_line(text):
    text = text.strip()
    return bool(re.search(
        # check for Part I/3.2+Company Overview & Mission+5
        r"^(Part\s+[IVXLC]+|[\d]+\.[\d]+)\s+.*\s+\d+$",
        text,
        re.IGNORECASE
    ))

# clean the documents by removing unwanted data
def clean_documents(docs):
    cleaned_docs = []
    skip_toc = False
    removed = 0

    for doc in docs:
        text = doc.page_content.strip()
        lower = text.lower()
        category = doc.metadata.get("category", "")

        # Normalize
        text = re.sub(r"\s+", " ", text)

        # Remove header/footer via metadata
        if category in ["Header", "Footer"]:
            removed += 1
            continue

        # Regex-based removal
        if is_header_footer(text):
            removed += 1
            continue

        # Detect TOC start
        if "table of contents" in lower:
            skip_toc = True
            continue

        # Skip TOC content
        if skip_toc:
            if re.match(r"^1\.\s", text):   # first real section
                skip_toc = False
            else:
                continue

        # Tag type
        if category == "Table":
            doc.metadata["type"] = "table"
        else:
            doc.metadata["type"] = "text"

        # Save cleaned content
        doc.page_content = text
        cleaned_docs.append(doc)

    print(f"Removed {removed} noisy chunks")
    print(f"Remaining chunks: {len(cleaned_docs)}")

    return cleaned_docs

In [14]:
docs_clean = clean_documents(docs)

Removed 21 noisy chunks
Remaining chunks: 460


In [15]:
# check if cleaning worked or not
print(docs[10])
print(docs_clean[10])

page_content='Tel: +1 (415) 700-9000 | hr@nexacorp.com | www.nexacorp.com' metadata={'source': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'category_depth': 0, 'filename': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'last_modified': '2026-05-01T11:30:41', 'page_number': 1, 'languages': ['eng'], 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'parent_id': '455472029b5887c8acdc6c539fd7b95a', 'category': 'UncategorizedText', 'element_id': '96a4b1466765e57ee816307d4291b752', 'type': 'text'}
page_content='' metadata={'source': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'languages': ['eng'], 'filename': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'category': 'PageBreak', 'element_id': '374c9b677aede328ee9f4fdefb686eb5', 'type': 'text'}


**Create Structured document**

In [16]:
def build_structured_docs(docs_clean):
    structured_docs = []
    current_part = ""
    current_section = ""
    current_subsection = ""
    buffer = []

    def flush_buffer(doc_type="text"):
        if buffer:
            structured_docs.append({
                "part": current_part,
                "section": current_section,
                "subsection": current_subsection,
                "content": " ".join(buffer).strip(),
                "type": doc_type
            })
            buffer.clear()

    for doc in docs_clean:
        text = re.sub(r"\s+", " ", doc.page_content).strip()
        category = doc.metadata.get("category", "")
        
        if not text:
            continue

        # HEADING DETECTION
        if category == "Title":
            # save previous content
            flush_buffer()

            # LEVEL 3
            if re.match(r"^\d+\.\d+\.\d+\s+", text):
                current_subsection = text

            # LEVEL 2
            elif re.match(r"^\d+\.\d+\s+", text):
                current_section = text
                current_subsection = ""

            # LEVEL 1
            elif re.match(r"^\d+\.\s+", text):
                current_part = text
                current_section = ""
                current_subsection = ""
            continue

        # TABLE HANDLING
        if (
            doc.metadata.get("type") == "table"
            or category == "Table"
        ):
            flush_buffer()
            structured_docs.append({
                "part": current_part,
                "section": current_section,
                "subsection": current_subsection,
                "content": text,
                "type": "table"
            })
            continue

        # NORMAL TEXT
        buffer.append(text)

    # final 
    flush_buffer()
    print(f"Created {len(structured_docs)} structured documents")
    return structured_docs

In [17]:
structured_docs = build_structured_docs(docs_clean)

Created 136 structured documents


In [18]:
print(structured_docs)

[{'part': '', 'section': '', 'subsection': '', 'content': 'NEXACORP GLOBAL Document Owner: Chief People Officer & General Counsel Next Review Date: December 31, 2025 NexaCorp Global Headquarters One NexaTower, 500 Innovation Drive, San Francisco, CA 94105 Tel: +1 (415) 700-9000 | hr@nexacorp.com | www.nexacorp.com', 'type': 'text'}, {'part': '', 'section': '', 'subsection': '', 'content': 'This handbook supersedes all previously issued policy documents, memos, and guidelines. In the event of a conflict between this document and any country-specific addendum, the addendum shall prevail to the extent required by local law. IMPORTANT: Employees are required to acknowledge receipt and understanding of this handbook annually via the HR Information System (HRIS) portal. Failure to complete the annual acknowledgement may result in access restrictions to certain company systems.', 'type': 'text'}, {'part': '1. Company Overview & Mission', 'section': '1.1 Corporate History & Identity', 'subsect

**Create Parent Documents**

In [19]:
parent_docs = []

for item in structured_docs:   
    doc_id = str(uuid.uuid4())
    # Add hierarchy context into content 
    full_content = f"""
        Part: {item.get("part", "")}
        Section: {item.get("section", "")}
        Subsection: {item.get("subsection", "")}

        {item["content"]}
        """.strip()

    parent_docs.append(
        Document(
            page_content=full_content,
            metadata={
                "doc_id": doc_id,
                "part": item.get("part", ""),
                "section": item.get("section", ""),
                "subsection": item.get("subsection", ""),
                "raw_content": item["content"],
                "type": item.get("type", "text")
            }
        )
    )

print(f"Created {len(parent_docs)} parent documents")

Created 136 parent documents


**BM25 Retriever**

In [20]:
bm25_retriever = BM25Retriever.from_documents(parent_docs)
bm25_retriever.k = 4

**Enbedding and Vector Store**

In [21]:
# create embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"},   
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 32
    }
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1258.64it/s]


In [22]:
# create FAISS vector store
dimension = 768

# use cosine similarity (IMPORTANT)
index = faiss.IndexFlatIP(dimension)

vectorstore = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

**Parent Document Retriever**

In [23]:
store = InMemoryStore()

parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,

    # REQUIRED
    child_splitter=RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    ),

    # OPTIONAL but recommended
    parent_splitter=RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=200
    ),
)

# ONLY pass parent docs
parent_retriever.add_documents(parent_docs)

In [24]:
parent_retriever.invoke("leave policy")

[Document(metadata={'doc_id': 'af6d477e-5149-4c63-8282-6b5bd2c8349e', 'part': '3. Human Resources Policies', 'section': '3.7 Leave Management Policy', 'subsection': '3.7.2 Paid Holidays', 'raw_content': "Holiday Typical Date Notes New Year's Day January 1 Observed on nearest weekday if falls on weekend Martin Luther King Jr. Day Third Monday in January Presidents' Day Third Monday in February Memorial Day Last Monday in May Juneteenth National Independence Day June 19 Independence Day July 4 Labor Day First Monday in September Thanksgiving Day Fourth Thursday in November Day After Thanksgiving Friday following Thanksgiving NexaCorp-designated day Christmas Day December 25 NexaDay (Floating Holiday) Employee's choice May not be carried over; use by Dec 31", 'type': 'table'}, page_content="Part: 3. Human Resources Policies\n        Section: 3.7 Leave Management Policy\n        Subsection: 3.7.2 Paid Holidays\n\n        Holiday Typical Date Notes New Year's Day January 1 Observed on neare

**Create Hybrid Retriever**

In [25]:
vector_retriever = parent_retriever

In [26]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6]
)

**Reranking retrieved documents**

In [27]:
reranker_model = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-base"
)

compressor = CrossEncoderReranker(model=reranker_model)

compression_retriever = ContextualCompressionRetriever(
    base_retriever=hybrid_retriever,
    base_compressor=compressor
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1695.93it/s]


testing retriever

In [28]:
docs = compression_retriever.invoke("leave policy")
docs

[Document(metadata={'doc_id': 'a93ce380-f120-4125-9fd2-fab6191d6bdb', 'part': '3. Human Resources Policies', 'section': '3.7 Leave Management Policy', 'subsection': '3.7.4 Sick Leave — State-Specific Accruals', 'raw_content': "In jurisdictions where state or local law mandates separate sick leave accruals (e.g., California, New York, New Jersey, Colorado, Washington), NexaCorp's unified PTO bank satisfies and exceeds the statutory minimum in all cases. Employees in these states have the right to use PTO for qualifying sick leave purposes without any separate approval process. HR Operations maintains a current map of state-specific sick leave requirements which is updated quarterly.", 'type': 'text'}, page_content="Part: 3. Human Resources Policies\n        Section: 3.7 Leave Management Policy\n        Subsection: 3.7.4 Sick Leave — State-Specific Accruals\n\n        In jurisdictions where state or local law mandates separate sick leave accruals (e.g., California, New York, New Jersey, 

In [29]:
docs = compression_retriever.invoke("company values")
docs

[Document(metadata={'doc_id': 'c42569a3-eed4-42e5-a496-f86aac0c2ba7', 'part': '1. Company Overview & Mission', 'section': '1.2 Vision, Mission & Core Values', 'subsection': '1.2.3 Core Values', 'raw_content': 'Value What It Means in Practice Integrity First We act honestly and transparently even when it is difficult. We do not misrepresent data, timelines, or capabilities to clients, colleagues, or regulators. Customer Obsession Every decision is evaluated through the lens of long-term customer value. Short-term revenue that compromises customer trust is never acceptable. Bold Innovation We reward calculated risk-taking and intelligent failure. Bureaucracy that impedes experimentation must be challenged through proper channels. Inclusive Excellence Diversity is not a compliance exercise — it is a competitive advantage. We build teams that reflect the global communities we serve. Ownership Mindset Every employee takes responsibility for outcomes, not just activities. We do not hide behi

In [30]:
docs = compression_retriever.invoke("organisational structure")
docs

[Document(metadata={'doc_id': '6a2a76e1-512b-4229-adb3-ccb72927cf8e', 'part': '1. Company Overview & Mission', 'section': '1.3 Organisational Structure', 'subsection': '1.3.1 Executive Leadership Team (ELT)', 'raw_content': 'The ELT is responsible for setting strategic direction, allocating capital, managing enterprise risk, and ensuring regulatory compliance. The ELT convenes monthly in person and weekly via video conference. All ELT members serve at the pleasure of the Board of Directors.', 'type': 'text'}, page_content='Part: 1. Company Overview & Mission\n        Section: 1.3 Organisational Structure\n        Subsection: 1.3.1 Executive Leadership Team (ELT)\n\n        The ELT is responsible for setting strategic direction, allocating capital, managing enterprise risk, and ensuring regulatory compliance. The ELT convenes monthly in person and weekly via video conference. All ELT members serve at the pleasure of the Board of Directors.'),
 Document(metadata={'doc_id': '104be773-2c70

**Query Decomposition**

In [31]:
llm = init_chat_model("google_genai:gemini-2.5-flash-lite")

In [32]:
from pydantic import BaseModel, Field
from typing import List, Literal
from langchain_core.prompts import PromptTemplate

# Structured Output Schema
class QueryAnalysis(BaseModel):
    needs_decomposition: Literal["yes", "no"] = Field(description="Whether the query should be decomposed into multiple subqueries")
    subqueries: List[str] = Field(description="List of generated subqueries")

# Prompt
decomposition_prompt = PromptTemplate(
    template=""" You are a query analysis assistant for a Retrieval-Augmented Generation (RAG) chatbot. Your task is to determine whether the given query should be decomposed into multiple subqueries.

Decompose the query if:
- it asks about more than one topic
- it contains words like "and", "or", "compare", "vs"
- it requires multi-hop reasoning
- it combines unrelated concepts

If decomposition is needed:
- set needs_decomposition = "yes"
- generate independent standalone subqueries

If decomposition is not needed:
- set needs_decomposition = "no"
- return an empty subqueries list

Rules:
- Preserve the original meaning
- Keep subqueries short and retrieval-friendly
- Do not answer the query
- Only return structured output

User Query:
{query}
""",
    input_variables=["query"]
)

structured_llm = llm.with_structured_output(QueryAnalysis)
decomposition_chain = decomposition_prompt | structured_llm

# Function to analyze query
def analyze_query(query: str):
    result = decomposition_chain.invoke({"query": query})
    if result.needs_decomposition == "no":
        return [query]
    return result.subqueries

query = "Explain company values and organisational structure"
final_queries = analyze_query(query)
print(final_queries)

['Explain company values', 'Explain organisational structure']


In [35]:
all_docs = []

for q in final_queries:
    docs = compression_retriever.invoke(q)
    all_docs.extend(docs)

In [36]:
all_docs

[Document(metadata={'doc_id': 'c42569a3-eed4-42e5-a496-f86aac0c2ba7', 'part': '1. Company Overview & Mission', 'section': '1.2 Vision, Mission & Core Values', 'subsection': '1.2.3 Core Values', 'raw_content': 'Value What It Means in Practice Integrity First We act honestly and transparently even when it is difficult. We do not misrepresent data, timelines, or capabilities to clients, colleagues, or regulators. Customer Obsession Every decision is evaluated through the lens of long-term customer value. Short-term revenue that compromises customer trust is never acceptable. Bold Innovation We reward calculated risk-taking and intelligent failure. Bureaucracy that impedes experimentation must be challenged through proper channels. Inclusive Excellence Diversity is not a compliance exercise — it is a competitive advantage. We build teams that reflect the global communities we serve. Ownership Mindset Every employee takes responsibility for outcomes, not just activities. We do not hide behi

**JSON Output**

In [33]:
def retrieve_json(query, retriever):
    docs = retriever.invoke(query)
    results = []
    for doc in docs:
        results.append({
            "part": doc.metadata.get("part"),
            "section": doc.metadata.get("section"),
            "subsection": doc.metadata.get("subsection"),
            "type": doc.metadata.get("type"),
            "content": doc.metadata.get("raw_content")
            
        })
    return results

In [34]:
results = retrieve_json(
    "Explain organisational structure",
    compression_retriever
)
print(json.dumps(results, indent=4))

[
    {
        "part": "1. Company Overview & Mission",
        "section": "1.3 Organisational Structure",
        "subsection": "1.3.1 Executive Leadership Team (ELT)",
        "type": "text",
        "content": "The ELT is responsible for setting strategic direction, allocating capital, managing enterprise risk, and ensuring regulatory compliance. The ELT convenes monthly in person and weekly via video conference. All ELT members serve at the pleasure of the Board of Directors."
    },
    {
        "part": "1. Company Overview & Mission",
        "section": "1.3 Organisational Structure",
        "subsection": "1.3.2 Governance Committees",
        "type": "text",
        "content": "In addition to the ELT, the following standing committees provide oversight and governance across key domains: Audit & Risk Committee (ARC): Reports to the Board; oversees financial controls, internal audit, and enterprise risk management. Ethics & Compliance Committee (ECC): Chaired by the CLO; overse